# Notebook 02: Building the Feature Function

In notebook 01, I calculated distance, gain, etc. as loose code tied to one trail. Here I **consolidate that into reusable functions** so the exact same logic can run on 1 trail today or 100 trails later, and inside the web app when a user uploads their own GPX.

I'll write three functions, each with one clear job:

| Function | Job |
|----------|-----|
| `load_trail(path)` | read a `.gpx` file → tidy DataFrame |
| `haversine(...)` | distance in metres between two lat/lon points |
| `compute_features(df)` | DataFrame → dictionary of difficulty features |

Splitting *loading* from *computing* matters: trails might come from different sources (Wikiloc, a user upload), but once they're a DataFrame, the feature logic is identical → **separation of concerns**.

In [1]:
import glob
import gpxpy
import numpy as np
import pandas as pd

## 1. `load_trail`

Same parsing as notebook 01, just wrapped in a function so I can call it on any file path.

In [2]:
def load_trail(path):
    """Read a GPX file and return a DataFrame of its track points."""
    with open(path, 'r', encoding='utf-8') as f:
        gpx = gpxpy.parse(f)
    points = gpx.tracks[0].segments[0].points
    return pd.DataFrame([
        {'lat': p.latitude, 'lon': p.longitude, 'elevation_m': p.elevation}
        for p in points
    ])

## 2. `haversine`

Identical to notebook 01. It turns two lat/lon coordinates into metres on the ground.

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    """Ground distance in metres between two lat/lon points."""
    R = 6_371_000  # Earth's radius in metres
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

## 3. `compute_features`

This takes one trail's DataFrame and returns a dictionary of numbers describing how hard it is. Each line is a feature the model can later learn from.

**The features, and why each one signals difficulty:**

| Feature | Meaning | Why it matters |
|---------|---------|----------------|
| `distance_km` | total length | longer = more endurance |
| `elevation_gain_m` | total climbing (smoothed) | trail's difficulty driver |
| `elevation_loss_m` | total descent | hard on the knees; steep downhills are tiring |
| `gain_rate_m_per_km` | climbing per km | steepness summary |
| `max/min/range elevation` | height stats | altitude & overall climb |
| `mean_slope_deg` | average steepness | sustained effort |
| `max_slope_deg` | steepest bit | scrambles / very hard sections |
| `slope_variability` | how much slope varies | rough, uneven terrain |
| `pct_steep_segments` | % of trail steeper than 20° | how much is genuinely steep |
| `exposure_index` | fraction high on the ridge | exposed, weather-prone ground |

**Key detail:** I smooth the elevation (3-point rolling average) *before* measuring gain and slope, fixing the GPS-noise inflation I found in notebook 01. Min/max elevation use the raw values (those were already exact).

In [4]:
def compute_features(df, smooth_window=3):
    """Turn a trail DataFrame (lat, lon, elevation_m) into difficulty features."""
    d = df.copy()

    # Smooth elevation to remove GPS jitter, then measure changes on the smoothed line
    d['elev_smooth'] = d['elevation_m'].rolling(smooth_window, center=True, min_periods=1).mean()
    d['elev_change'] = d['elev_smooth'].diff().fillna(0)

    # Horizontal distance between consecutive points
    d['step_m'] = haversine(d['lat'].shift(), d['lon'].shift(), d['lat'], d['lon']).fillna(0)

    # Slope of each segment in degrees (avoid divide-by-zero on stationary points)
    horiz = d['step_m'].replace(0, np.nan)
    slope = np.degrees(np.arctan(d['elev_change'] / horiz)).fillna(0).abs()

    dist_km = d['step_m'].sum() / 1000
    gain = d.loc[d['elev_change'] > 0, 'elev_change'].sum()
    loss = -d.loc[d['elev_change'] < 0, 'elev_change'].sum()

    elev = d['elevation_m']
    elev_range = elev.max() - elev.min()
    exposure = (elev >= elev.min() + 0.8 * elev_range).mean()

    return {
        'distance_km': round(dist_km, 2),
        'elevation_gain_m': round(gain),
        'elevation_loss_m': round(loss),
        'gain_rate_m_per_km': round(gain / dist_km, 1),
        'max_elevation_m': round(elev.max()),
        'min_elevation_m': round(elev.min()),
        'elevation_range_m': round(elev_range),
        'mean_slope_deg': round(slope.mean(), 1),
        'max_slope_deg': round(slope.max(), 1),
        'slope_variability': round(slope.std(), 1),
        'pct_steep_segments': round((slope > 20).mean() * 100, 1),
        'exposure_index': round(exposure, 2),
    }

## 4. Test it on the Dolomites trail

The whole point: three short function calls now do everything notebook 01 did line-by-line.

In [5]:
path = glob.glob('../data/raw/gpx/*.gpx')[0]
df = load_trail(path)
features = compute_features(df)

# Show as a tidy table
pd.Series(features).to_frame('value')

,value
distance_km,13.48
elevation_gain_m,1301.00
elevation_loss_m,1301.00
gain_rate_m_per_km,96.50
max_elevation_m,1847.00
min_elevation_m,792.00
elevation_range_m,1055.00
mean_slope_deg,13.40
max_slope_deg,61.00
slope_variability,9.90


## 5. Sanity check

- `distance_km` ≈ 13.5
- `elevation_gain_m` ≈ 1,301 (matches Wikiloc's 1,333 now that we smooth)
- `max/min elevation` = 1,847 / 792 (exact)

Everything lines up.

## 6. What's next

1. **Scale up:** once more trails are downloaded, loop `load_trail` + `compute_features` over all of them to build a feature table (one row per trail).
2. **Graduate the code:** move these three functions out of the notebook into a `src/features.py` file, so both the modelling code *and* the web app can import and reuse them.